# One Bridge Ruins Single Linkage

**Topic:** Linkage criteria &nbsp;·&nbsp; **Difficulty:** Medium

Single linkage measures the distance between two clusters by their *closest* pair
of points. That makes it wonderful at following thin curved shapes and
catastrophic when a thin line of noise connects two groups that should stay
apart: the chain of near-touching points welds them together.

Two well-separated blobs, one sparse bridge of 12 points between them. Watch
single and complete linkage disagree completely.

## The data (created in the next cell)

- `X` `(412, 2)`: two blobs of 200 points each plus a 12-point bridge.
- `y_true` `(412,)`: blob membership, with bridge points labelled `2`.

## Your task

With **sklearn and NumPy**:

1. Fit `AgglomerativeClustering(n_clusters=2, linkage=...)` for both
   `"single"` and `"complete"`.
2. `blob_only`: the mask selecting the 400 non-bridge points.
3. For each linkage, compute the adjusted Rand index **on the blob points only**,
   against their true blob labels.
4. Also record `size_split_single` and `size_split_complete`, the size of the
   smaller of the two clusters found, over all 412 points.
5. Set `answers` to a dict with `"ari_single"`, `"ari_complete"`,
   `"size_split_single"`, `"size_split_complete"`.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks complete linkage recovers the blobs while single linkage does not.

In [ ]:
import numpy as np

rng = np.random.default_rng(1818)
a = rng.normal([0, 0], 0.6, (200, 2))
b = rng.normal([9, 0], 0.6, (200, 2))
t = np.linspace(0.12, 0.88, 12)
bridge = np.column_stack([t * 9.0, rng.normal(0, 0.05, 12)])
X = np.vstack([a, b, bridge])
y_true = np.concatenate([np.zeros(200, int), np.ones(200, int),
                         np.full(12, 2, int)])
print(f"X {X.shape}: 200 + 200 blob points and a {len(bridge)}-point bridge")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("ari_single", [], 0.0, 0.0, 1e-06),
    ("size_split_single", [], 1.0, -0.16743801481449916, 1e-06),
    ("ari_complete", [], 1.0, -0.16743801481449916, 1e-06),
    ("size_split_complete", [], 204.0, -34.15735502215783, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "ari_single" in answers and "ari_complete" in answers:
    _w = answers["ari_complete"] > 0.9 and answers["ari_single"] < 0.2
    print(f"  {'chaining breaks single':<24} {'ok' if _w else 'wrong'}  "
          f"(single {answers['ari_single']:.3f}, complete {answers['ari_complete']:.3f})")
    _ok = _ok and _w

print("=" * 56)
if _ok:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

The bridge points are each about 0.62 apart, which is smaller than the width of
either blob, so single linkage merges the whole chain before it ever considers
separating the blobs, and the "two clusters" it reports are one giant component
plus a single outlier. Complete linkage uses the *farthest* pair, so a thin chain
buys nothing and the blobs stay apart. The trade is that complete linkage cannot
follow genuinely elongated shapes, which is exactly what single linkage is
for.

```python
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score

blob_only = y_true != 2

out = {}
for link in ("single", "complete"):
    lab = AgglomerativeClustering(n_clusters=2, linkage=link).fit_predict(X)
    out[f"ari_{link}"] = float(adjusted_rand_score(y_true[blob_only],
                                                   lab[blob_only]))
    counts = np.bincount(lab.astype(np.intp), minlength=2)
    out[f"size_split_{link}"] = int(counts.min())

answers = out
```